In [1]:
# Cell 1
%%capture
!pip install -q --upgrade transformers
!pip install -q sentence-transformers chromadb langchain-text-splitters groq rouge-score colorama

In [2]:
import re, time
from collections import defaultdict

from groq import Groq
from colorama import init as colorama_init
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rouge_score import rouge_scorer as rouge_lib
import chromadb
import numpy as np

colorama_init(autoreset=True)

GDRIVE_FILE_PATH = "/content/drive/MyDrive/budha_english.txt"

try:
    from google.colab import userdata
    HF_TOKEN     = userdata.get("HF_TOKEN") or ""
    GROQ_API_KEY = userdata.get("GROQ_API_KEY") or ""
except Exception:
    HF_TOKEN     = ""
    GROQ_API_KEY = ""

if not HF_TOKEN:
    print("Warning: HF_TOKEN not set — only public models can be loaded")
if not GROQ_API_KEY:
    print("Warning: GROQ_API_KEY not set — generation calls will fail")

EMBEDDING_MODEL   = "microsoft/harrier-oss-v1-0.6b"
GROQ_MODEL        = "llama-3.3-70b-versatile"
CHROMA_COLLECTION = "kakatarua_harrier_rag_v1"
TOP_K             = 5

RESEARCH_TOPICS = [
    "Budha's patriotism and role in the Bangladesh Liberation War",
    "Budha's character development from orphan to freedom fighter",
    "The role of loneliness and independence in shaping Budha's identity",
    "Selina Hossain's narrative portrayal of violence and resilience in Kakatarua",
]

print(f"Embedding model  : {EMBEDDING_MODEL}")
print(f"LLM              : {GROQ_MODEL}")
print(f"ChromaDB         : {CHROMA_COLLECTION}")
print(f"Research topics  : {len(RESEARCH_TOPICS)}")

Embedding model  : microsoft/harrier-oss-v1-0.6b
LLM              : llama-3.3-70b-versatile
ChromaDB         : kakatarua_harrier_rag_v1
Research topics  : 4


In [3]:
# Cell 3
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

with open(GDRIVE_FILE_PATH, "r", encoding="utf-8") as fh:
    raw_text = fh.read()

print(f"Loaded  : {len(raw_text):,} characters")
print(f"Preview : {raw_text[:250]!r}")


Mounted at /content/drive
Loaded  : 87,053 characters
Preview : 'Novel : Introduction a. Concept and Definition of Novel A novel is a story written in prose. People love to tell stories. It has been going on since ancient times. People used to tell stories by word of mouth There was no way to write it down. Later,'


In [4]:
def clean_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    lines = text.split("\n")
    kept = []
    for line in lines:
        s = line.strip()
        if len(s) >= 4 or s == "":
            kept.append(s)
    text = "\n".join(kept)
    text = re.sub(r" {2,}", " ", text)
    return text.strip()

text_content = clean_text(raw_text)
removed = len(raw_text) - len(text_content)
print(f"Before : {len(raw_text):,} chars")
print(f"After  : {len(text_content):,} chars  ({removed:,} chars removed)")

Before : 87,053 chars
After  : 87,042 chars  (11 chars removed)


In [5]:
# Cell 5
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=1500,
    chunk_overlap=150,
    length_function=len,
)

documents = splitter.split_text(text_content)
avg_len = sum(len(d) for d in documents) / len(documents)
print(f"Chunks : {len(documents)}")
print(f"Avg len: {avg_len:.0f} chars")
print(f"\nSample chunk 0:\n{documents[0][:400]}")

Chunks : 89
Avg len: 1009 chars

Sample chunk 0:
Novel : Introduction a. Concept and Definition of Novel A novel is a story written in prose. People love to tell stories. It has been going on since ancient times. People used to tell stories by word of mouth There was no way to write it down. Later, the novel appeared in its continuation. But the novel is not just a story, it is a kind of creative work. Human life is the source of novels. Writers


In [6]:
# Cell 6
print("Loading embedding model …")
embed_model = SentenceTransformer(
    EMBEDDING_MODEL,
    model_kwargs={"dtype": "auto"},
    token=HF_TOKEN or None,
)

print("Encoding chunks …")
doc_embeddings = embed_model.encode(documents, show_progress_bar=True, batch_size=32)
print(f"Embedding matrix : {doc_embeddings.shape}")

chroma_client = chromadb.EphemeralClient()
try:
    chroma_client.delete_collection(CHROMA_COLLECTION)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=CHROMA_COLLECTION,
    metadata={"hnsw:space": "cosine"},
)
collection.add(
    documents=documents,
    embeddings=doc_embeddings.tolist(),
    ids=[str(i) for i in range(len(documents))],
)
print(f"ChromaDB ready   : {collection.count()} vectors in '{CHROMA_COLLECTION}'")

Loading embedding model …


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/7.61k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

...

chat_template.jinja:   0%|          | 0.00/4.12k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Encoding chunks …


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embedding matrix : (89, 1024)
ChromaDB ready   : 89 vectors in 'kakatarua_harrier_rag_v1'


In [7]:
# Cell 7
def retrieve(query: str, top_k: int = TOP_K) -> list:
    """Return the top-k document chunks most similar to *query* (cosine distance)."""
    # Harrier is an asymmetric embedding model: queries need an instruction prompt
    # (per the model card, omitting it degrades retrieval); documents are encoded plain.
    q_vec = embed_model.encode([query], prompt_name="web_search_query")
    results = collection.query(
        query_embeddings=q_vec.tolist(),
        n_results=top_k,
    )
    return results["documents"][0]

def context_for(topic: str, top_k: int = TOP_K) -> str:
    """Retrieve and join top-k chunks for *topic* into a single context string."""
    return "\n\n".join(retrieve(topic, top_k=top_k))

sample_chunks = retrieve(RESEARCH_TOPICS[0], top_k=2)
print(f"Retrieval smoke-test — top 2 chunks for: '{RESEARCH_TOPICS[0]}'")
for i, c in enumerate(sample_chunks, 1):
    print(f"\n[Rank {i}] {c[:200]}…")

Retrieval smoke-test — top 2 chunks for: 'Budha's patriotism and role in the Bangladesh Liberation War'

[Rank 1] This is the story of Budha, the main character of Kaktaduya novel. After that Selina Hossain gradually developed the character of Budha. Although Budha is a teenager, he possesses infinite courage and…

[Rank 2] The whole village and the bazaar became his haunt. All the people he knew became his own. This is how the day goes, the day comes. But one day the military entered the village. Burned the shops in the…


In [8]:
# Cell 8
ICL_EXAMPLES = [
    {
        "q": "What does Budha's nickname 'Kakatarua' (scarecrow) signify about his character?",
        "a": "Like a scarecrow he stands alone and unnoticed, watching over the village — a guise of stillness that hides his courage and lets him strike at the occupying army.",
    },
    {
        "q": "How does Selina Hossain use Budha's early trauma to motivate his later militancy?",
        "a": "Losing his entire family to cholera and living alone removes childhood fear, making armed resistance feel natural rather than extreme.",
    },
    {
        "q": "What narrative function do the Razakar collaborators serve in the novel?",
        "a": "They embody the local face of oppression, grounding the liberation war at the village level and giving Budha a concrete, immediate enemy.",
    },
]

_TASK = (
    'Generate exactly 5 short-answer questions with concise one-to-two-sentence answers '
    'strictly grounded in the context provided above. '
    'Focus on the topic: "{topic}". '
    'Format each pair as:\nQ: <question>\nA: <answer>\n'
)

def _fmt_example(ex: dict) -> str:
    return f"Q: {ex['q']}\nA: {ex['a']}"

def build_zero_shot_prompt(context: str, topic: str) -> str:
    return f"Context:\n{context}\n\n" + _TASK.format(topic=topic)

def build_one_shot_prompt(context: str, topic: str) -> str:
    return (
        f"Context:\n{context}\n\n"
        f"Example:\n{_fmt_example(ICL_EXAMPLES[0])}\n\n"
        + _TASK.format(topic=topic)
    )

def build_few_shot_prompt(context: str, topic: str) -> str:
    examples = "\n\n".join(_fmt_example(ex) for ex in ICL_EXAMPLES)
    return (
        f"Context:\n{context}\n\n"
        f"Examples:\n{examples}\n\n"
        + _TASK.format(topic=topic)
    )

PROMPT_BUILDERS = {
    "zero_shot": build_zero_shot_prompt,
    "one_shot":  build_one_shot_prompt,
    "few_shot":  build_few_shot_prompt,
}

print("Prompt builders:", list(PROMPT_BUILDERS))
_demo_ctx = "Budha is the main character…"
print("\n── Zero-shot prompt sample ──")
print(build_zero_shot_prompt(_demo_ctx, RESEARCH_TOPICS[0]))

Prompt builders: ['zero_shot', 'one_shot', 'few_shot']

── Zero-shot prompt sample ──
Context:
Budha is the main character…

Generate exactly 5 short-answer questions with concise one-to-two-sentence answers strictly grounded in the context provided above. Focus on the topic: "Budha's patriotism and role in the Bangladesh Liberation War". Format each pair as:
Q: <question>
A: <answer>



In [9]:
groq_client = Groq(api_key=GROQ_API_KEY)

def generate(prompt: str, temperature: float = 0.3, max_tokens: int = 1024, retries: int = 3) -> str:
    """Send *prompt* to Groq and return the assistant response text, with exponential-backoff retry."""
    for attempt in range(1, retries + 1):
        try:
            completion = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return completion.choices[0].message.content.strip()
        except Exception as e:
            if attempt == retries:
                raise
            wait = 2 ** attempt
            print(f"  Groq error (attempt {attempt}/{retries}): {e}. Retrying in {wait}s…")
            time.sleep(wait)

_test = generate("Reply with exactly: OK")
print(f"Groq smoke-test: {_test!r}")
print(f"Model: {GROQ_MODEL}")

Groq smoke-test: 'OK'
Model: llama-3.3-70b-versatile


In [10]:
results  = defaultdict(dict)
contexts = {}  # keyed by topic; reused in Cell 11 to avoid re-embedding

for topic in RESEARCH_TOPICS:
    print(f"\n{'='*68}")
    print(f"TOPIC: {topic}")
    print('='*68)
    ctx = context_for(topic)
    contexts[topic] = ctx

    for strategy, builder in PROMPT_BUILDERS.items():
        prompt  = builder(ctx, topic)
        output  = generate(prompt)
        results[topic][strategy] = output
        label   = strategy.upper().replace("_", "-")
        print(f"\n── {label} ──")
        print(output)
        time.sleep(1)

print(f"\n{'='*68}")
print("Experiment complete. All results stored in `results`.")


TOPIC: Budha's patriotism and role in the Bangladesh Liberation War

── ZERO-SHOT ──
Q: What motivated Budha to become a freedom fighter in the Bangladesh Liberation War?
A: Budha's motivation to become a freedom fighter stemmed from his compassion for the people of Bangladesh, hatred for foreign militaries, and patriotism, which drove him to take action against the Pakistani military that had entered his village and burned down shops.

Q: How did Budha's character develop as he grew up, and what impact did it have on his role in the war?
A: Budha grew up as a fearless and independent teenager, having lived alone, which made his involvement in the liberation war a natural progression, allowing him to take bold actions such as burning down the peace committee and Razaka commander's house.

Q: What was Budha's first major task as a freedom fighter, and how did he accomplish it?
A: Budha's first major task was to blow up the military camp of the school, which he accomplished by digging a

In [11]:
_scorer = rouge_lib.RougeScorer(["rougeL"], use_stemmer=True)

def extract_answers(text: str) -> list:
    """Collect answer text after each 'A:' line, joining any continuation lines."""
    answers, current = [], []
    for ln in text.split("\n"):
        if ln.startswith("A:"):
            if current:  # flush previous answer even without an intervening 'Q:'
                answers.append(" ".join(current))
            current = [ln[2:].strip()]
        elif ln.startswith("Q:"):
            if current:
                answers.append(" ".join(current))
                current = []
        elif current:
            stripped = ln.strip()
            if stripped:
                current.append(stripped)
    if current:
        answers.append(" ".join(current))
    return answers

def avg_word_count(text: str) -> float:
    answers = extract_answers(text)
    if not answers:
        return float("nan")
    return sum(len(a.split()) for a in answers) / len(answers)

# ROUGE-L F-measure is symmetric (F = 2·LCS / (|ref|+|cand|)), so argument order does not
# affect the computed value. Convention here matches scorer API: (reference, candidate).
# Computes overlap vs. zero-shot output — measures cross-strategy divergence, not quality.
def rouge_l_vs(reference: str, candidate: str) -> float:
    return round(_scorer.score(reference, candidate)["rougeL"].fmeasure, 4)

def context_relevance(generated: str, context: str) -> float:
    vecs = embed_model.encode([generated, context])
    cos  = np.dot(vecs[0], vecs[1]) / (
        np.linalg.norm(vecs[0]) * np.linalg.norm(vecs[1])
    )
    return round(float(cos), 4)

rows = []
for topic in RESEARCH_TOPICS:
    ctx      = contexts[topic]   # reuse context cached in Cell 10
    baseline = results[topic]["zero_shot"]
    for strategy in PROMPT_BUILDERS:
        text = results[topic][strategy]
        rows.append({
            "topic"    : (topic[:46] + "…") if len(topic) > 47 else topic,
            "strategy" : strategy,
            "avg_words": round(avg_word_count(text), 1),
            "div_zs"   : rouge_l_vs(baseline, text) if strategy != "zero_shot" else "—",
            "ctx_sim"  : context_relevance(text, ctx),
        })

W = 49
# div(ZS): ROUGE-L overlap with zero-shot output — lower means more divergent, not worse.
hdr = f"{'Topic':<{W}} {'Strategy':<11} {'AvgWords':>9} {'div(ZS)':>8} {'CtxSim':>7}"
print(hdr)
print("─" * len(hdr))
for r in rows:
    rl = f"{r['div_zs']:>8}" if r["div_zs"] != "—" else f"{'—':>8}"
    print(
        f"{r['topic']:<{W}} {r['strategy']:<11} {r['avg_words']:>9} {rl} {r['ctx_sim']:>7}"
    )

Topic                                             Strategy     AvgWords  div(ZS)  CtxSim
────────────────────────────────────────────────────────────────────────────────────────
Budha's patriotism and role in the Bangladesh …   zero_shot        42.6        —  0.7337
Budha's patriotism and role in the Bangladesh …   one_shot         39.0   0.4046  0.8039
Budha's patriotism and role in the Bangladesh …   few_shot         35.8   0.3408  0.7711
Budha's character development from orphan to f…   zero_shot        35.4        —  0.7804
Budha's character development from orphan to f…   one_shot         29.4   0.4564  0.7716
Budha's character development from orphan to f…   few_shot         32.8   0.3197  0.7499
The role of loneliness and independence in sha…   zero_shot        32.0        —  0.7152
The role of loneliness and independence in sha…   one_shot         39.2   0.4134  0.7212
The role of loneliness and independence in sha…   few_shot         33.6   0.3125   0.707
Selina Hossain's narr

In [12]:
# Cell 12
for topic in RESEARCH_TOPICS:
    print(f"\n{'#'*70}")
    print(f"TOPIC: {topic}")
    for strategy in PROMPT_BUILDERS:
        label = strategy.upper().replace("_", "-")
        print(f"\n── {label} ──────────────────────────────────────────────────")
        print(results[topic][strategy])



######################################################################
TOPIC: Budha's patriotism and role in the Bangladesh Liberation War

── ZERO-SHOT ──────────────────────────────────────────────────
Q: What motivated Budha to become a freedom fighter in the Bangladesh Liberation War?
A: Budha's motivation to become a freedom fighter stemmed from his compassion for the people of Bangladesh, hatred for foreign militaries, and patriotism, which drove him to take action against the Pakistani military that had entered his village and burned down shops.

Q: How did Budha's character develop as he grew up, and what impact did it have on his role in the war?
A: Budha grew up as a fearless and independent teenager, having lived alone, which made his involvement in the liberation war a natural progression, allowing him to take bold actions such as burning down the peace committee and Razaka commander's house.

Q: What was Budha's first major task as a freedom fighter, and how did he accomp